# Week 3 - RDS and Cognito in place of Supabase

## Run this before Week 3 Day 2 Lab 2

The app on Lambda needs a database and a sign-in that it can reach over the internet. The course uses a Supabase project for both. This notebook makes them on AWS instead: Amazon RDS for PostgreSQL for the data and an Amazon Cognito user pool for the handlers' sign-in, in us-east-1 and tagged `label=FDE-lab` (`docs/PLAN_CONTEXT.md`, Phase 7).

It writes their addresses into `.env` as `AWS_DATABASE_URL`, `AWS_COGNITO_USER_POOL_ID` and `AWS_COGNITO_CLIENT_ID`. Lab 2 and Day 3 read them from there, so they deploy the app with RDS and Cognito and your laptop keeps its local Postgres and Supabase.

## 3 parts:

PART 1: The course role gains RDS, Cognito and a security group

PART 2: The database on RDS and the sign-in on Cognito

PART 3: Teardown, once you've finished Week 3's labs

## What it costs

The deployed app keeps its data in Amazon RDS for PostgreSQL and signs handlers in with Amazon Cognito, in place of a Supabase project (`docs/PLAN_CONTEXT.md`, Phase 7). Everything is in us-east-1 and tagged `label=FDE-lab`.

What it costs each day if everything is left running and nobody uses it. Prices from AWS's price list for us-east-1, checked on 2026-11-02:

| What | Price | A day |
|---|---|---|
| RDS `db.t4g.micro`, PostgreSQL, Single-AZ | $0.016 an hour | $0.38 |
| RDS storage, 20 GB gp3 | $0.115 per GB-month | $0.08 |
| RDS public IPv4 address | $0.005 an hour | $0.12 |
| The API key's secret in Secrets Manager (AgentCore Identity) | $0.40 a month | $0.01 |
| **Total while idle** | | **about $0.60 (about $18 a month)** |

Nothing while idle: Cognito (free for three users), the two Lambda functions (charged only when called), and, as far as we know, the Gateway, the Policy and Memory, which are charged per call. Those last three weren't checked against the price list. A lab session adds cents of AWS calls and a few cents of GPT-6 Luna.

The account's credits or the RDS free tier may cover some of this. `uv run monitor.py` shows the real spend. Part 3 at the end deletes everything, once you've finished Week 3's labs.


In [ ]:
import truststore

# Trust the certificates in the Windows certificate store, so HTTPS works behind a network that inspects traffic.
truststore.inject_into_ssl()

import json
import os
import secrets
import shutil
import subprocess
import time
from pathlib import Path
from urllib.parse import urlsplit

import boto3
import psycopg
from dotenv import load_dotenv, set_key, unset_key

ENV_FILE = Path("../../.env")
load_dotenv(ENV_FILE, override=True)

## Connect to AWS

boto3 uses the `[default]` profile from Day 1, which signs in as the `undercover-course` role in us-east-1. If the sign-in has run out, run `aws sso login` in a terminal.

In [ ]:
session = boto3.Session()
region = session.region_name
identity = session.client("sts").get_caller_identity()
account = identity["Account"]

print(f"Working as {identity['Arn']} in {region}")

In [ ]:
from botocore.exceptions import (
    ClientError,
    SSOTokenLoadError,
    TokenRetrievalError,
    UnauthorizedSSOTokenError,
)

COURSE_ROLE = "undercover-course"
DB_INSTANCE = "undercover-db"
DB_SECURITY_GROUP = "undercover-db-sg"
DB_USER = "undercover_admin"
USER_POOL_NAME = "undercover-handlers"
APP_CLIENT_NAME = "undercover-dashboard"
FDE_TAGS = {"project": "undercover", "label": "FDE-lab"}
FDE_TAG_LIST = [{"Key": key, "Value": value} for key, value in FDE_TAGS.items()]

## Part 1: The course role gains RDS, Cognito and a security group

The `undercover-course` role from Day 1 can't make a database, a user pool or a security group. These cells
add what it needs to its policy, as Day 1's Step 5 does, with your administrator profile
(`AWS_ADMIN_PROFILE` in `.env`). They read Day 1's `undercover-course-policy.json`, add the statements
below (each `Sid` starts `FdeLab`), and put the result on the permission set, leaving the file as it is,
so running them again puts the same policy.

Where AWS allows it, the new permissions are limited: RDS to databases named `undercover...`, and the
security group's changes to groups tagged `label=FDE-lab`. A Cognito user pool's ARN has its id, not its
name, so the Cognito permissions cover the account's user pools in us-east-1.

Everything these cells make is in us-east-1, named `undercover...` and tagged `project=undercover` and
`label=FDE-lab`. Part 3 deletes it.

In [ ]:
ADMIN_PROFILE = os.environ["AWS_ADMIN_PROFILE"]

admin = boto3.Session(profile_name=ADMIN_PROFILE, region_name="us-east-1")
try:
    admin_identity = admin.client("sts").get_caller_identity()
except (SSOTokenLoadError, TokenRetrievalError, UnauthorizedSSOTokenError):
    subprocess.run(  # noqa: S603 (a fixed command)
        [shutil.which("aws"), "sso", "login", "--profile", ADMIN_PROFILE], check=True
    )
    admin_identity = admin.client("sts").get_caller_identity()

print(f"Signed in as the administrator {admin_identity['Arn'].split('/')[-1]}")

In [ ]:
ARN = f"us-east-1:{account}"
SECURITY_GROUPS = f"arn:aws:ec2:{ARN}:security-group/*"

FDE_LAB_STATEMENTS = [
    {
        "Sid": "FdeLabRdsDatabase",
        "Effect": "Allow",
        "Action": "rds:*",
        "Resource": f"arn:aws:rds:{ARN}:db:undercover*",
    },
    {
        "Sid": "FdeLabRdsCreateWithDefaults",
        "Effect": "Allow",
        "Action": "rds:CreateDBInstance",
        "Resource": [
            f"arn:aws:rds:{ARN}:{kind}:*" for kind in ("subgrp", "pg", "og", "secgrp")
        ],
    },
    {
        "Sid": "FdeLabRdsBackups",
        "Effect": "Allow",
        "Action": "rds:DeleteDBInstanceAutomatedBackup",
        "Resource": f"arn:aws:rds:{ARN}:auto-backup:*",
    },
    {
        "Sid": "FdeLabRdsRead",
        "Effect": "Allow",
        "Action": ["rds:Describe*", "rds:ListTagsForResource"],
        "Resource": "*",
    },
    {
        "Sid": "FdeLabRdsServiceRole",
        "Effect": "Allow",
        "Action": "iam:CreateServiceLinkedRole",
        "Resource": "*",
        "Condition": {"StringEquals": {"iam:AWSServiceName": "rds.amazonaws.com"}},
    },
    {
        "Sid": "FdeLabRdsEncryption",
        "Effect": "Allow",
        "Action": ["kms:CreateGrant", "kms:DescribeKey"],
        "Resource": "*",
        "Condition": {
            "StringEquals": {"kms:ViaService": "rds.us-east-1.amazonaws.com"}
        },
    },
    {
        "Sid": "FdeLabCognitoCreate",
        "Effect": "Allow",
        "Action": ["cognito-idp:CreateUserPool", "cognito-idp:ListUserPools"],
        "Resource": "*",
    },
    {
        "Sid": "FdeLabCognitoPools",
        "Effect": "Allow",
        "Action": "cognito-idp:*",
        "Resource": f"arn:aws:cognito-idp:{ARN}:userpool/*",
    },
    {
        "Sid": "FdeLabSecurityGroupVpc",
        "Effect": "Allow",
        "Action": "ec2:CreateSecurityGroup",
        "Resource": f"arn:aws:ec2:{ARN}:vpc/*",
    },
    {
        "Sid": "FdeLabSecurityGroupCreate",
        "Effect": "Allow",
        "Action": "ec2:CreateSecurityGroup",
        "Resource": SECURITY_GROUPS,
        "Condition": {"StringEquals": {"aws:RequestTag/label": "FDE-lab"}},
    },
    {
        "Sid": "FdeLabSecurityGroupTagOnCreate",
        "Effect": "Allow",
        "Action": "ec2:CreateTags",
        "Resource": SECURITY_GROUPS,
        "Condition": {"StringEquals": {"ec2:CreateAction": "CreateSecurityGroup"}},
    },
    {
        "Sid": "FdeLabSecurityGroupChange",
        "Effect": "Allow",
        "Action": [
            "ec2:AuthorizeSecurityGroupIngress",
            "ec2:RevokeSecurityGroupIngress",
            "ec2:DeleteSecurityGroup",
        ],
        "Resource": SECURITY_GROUPS,
        "Condition": {"StringEquals": {"aws:ResourceTag/label": "FDE-lab"}},
    },
    {
        "Sid": "FdeLabEc2Read",
        "Effect": "Allow",
        "Action": [
            "ec2:DescribeSecurityGroups",
            "ec2:DescribeSecurityGroupRules",
            "ec2:DescribeVpcs",
            "ec2:DescribeSubnets",
        ],
        "Resource": "*",
    },
    {
        "Sid": "FdeLabTagListing",
        "Effect": "Allow",
        "Action": "tag:GetResources",
        "Resource": "*",
    },
]

In [ ]:
sso = admin.client("sso-admin")
instance_arn = sso.list_instances()["Instances"][0]["InstanceArn"]
permission_set_arn = next(
    arn
    for page in sso.get_paginator("list_permission_sets").paginate(
        InstanceArn=instance_arn
    )
    for arn in page["PermissionSets"]
    if sso.describe_permission_set(InstanceArn=instance_arn, PermissionSetArn=arn)[
        "PermissionSet"
    ]["Name"]
    == COURSE_ROLE
)

course_policy = json.loads(
    Path("undercover-course-policy.json").read_text(encoding="utf-8")
)
course_policy["Statement"] += FDE_LAB_STATEMENTS
sso.put_inline_policy_to_permission_set(
    InstanceArn=instance_arn,
    PermissionSetArn=permission_set_arn,
    InlinePolicy=json.dumps(course_policy),
)

# The role in the account changes only when the permission set is provisioned again.
request_id = sso.provision_permission_set(
    InstanceArn=instance_arn,
    PermissionSetArn=permission_set_arn,
    TargetType="AWS_ACCOUNT",
    TargetId=account,
)["PermissionSetProvisioningStatus"]["RequestId"]
while True:
    status = sso.describe_permission_set_provisioning_status(
        InstanceArn=instance_arn, ProvisionPermissionSetRequestId=request_id
    )["PermissionSetProvisioningStatus"]
    if status["Status"] != "IN_PROGRESS":
        break
    time.sleep(2)
if status["Status"] == "FAILED":
    raise RuntimeError(status["FailureReason"])
print(f"{COURSE_ROLE} can now use RDS, Cognito and the database's security group")

Back on the course role: this cell checks who you are working as, and that the role can now see RDS. If it
says `AccessDenied`, wait a minute for the new policy to reach the role and run it again.

In [ ]:
rds = session.client("rds")
ec2 = session.client("ec2")
cognito = session.client("cognito-idp")

print(f"Working as {session.client('sts').get_caller_identity()['Arn']}")
print(f"RDS instances in us-east-1: {len(rds.describe_db_instances()['DBInstances'])}")

## Part 2: The database on Amazon RDS

The deployed app keeps its own copy of the data in Amazon RDS for PostgreSQL 17: `db.t4g.micro`, 20 GB of
gp3 storage, one Availability Zone, backups kept for 1 day, named `undercover-db`. Your laptop keeps local
Postgres.

It is publicly accessible, because a Lambda function outside a VPC can't reach a private database without
a NAT gateway, which costs about $32 a month. So its security group opens port 5432 to the internet. What
protects it is the password and TLS: from PostgreSQL 15, RDS refuses a connection without TLS
(`rds.force_ssl`), and the address asks for it (`sslmode=require`).

The password is random. It goes only into `.env`, inside `AWS_DATABASE_URL`, and is never printed. The cell
saves it before it creates the database, so it is never lost. Creating the database takes 5 to 10 minutes.

In [ ]:
vpc_id = ec2.describe_vpcs(Filters=[{"Name": "is-default", "Values": ["true"]}])[
    "Vpcs"
][0]["VpcId"]
groups = ec2.describe_security_groups(
    Filters=[
        {"Name": "group-name", "Values": [DB_SECURITY_GROUP]},
        {"Name": "vpc-id", "Values": [vpc_id]},
    ]
)["SecurityGroups"]
if groups:
    db_group_id = groups[0]["GroupId"]
    print(f"Security group {DB_SECURITY_GROUP} already exists")
else:
    db_group_id = ec2.create_security_group(
        GroupName=DB_SECURITY_GROUP,
        Description="FDE-lab: Postgres for the UnderCover app on Lambda",
        VpcId=vpc_id,
        TagSpecifications=[{"ResourceType": "security-group", "Tags": FDE_TAG_LIST}],
    )["GroupId"]
    print(f"Created security group {DB_SECURITY_GROUP}")

postgres_from_anywhere = {
    "IpProtocol": "tcp",
    "FromPort": 5432,
    "ToPort": 5432,
    "IpRanges": [
        {
            "CidrIp": "0.0.0.0/0",
            "Description": "FDE-lab: Lambda outside a VPC has no fixed address",
        }
    ],
}
try:
    ec2.authorize_security_group_ingress(
        GroupId=db_group_id, IpPermissions=[postgres_from_anywhere]
    )
    print("Opened port 5432")
except ClientError as error:
    if error.response["Error"]["Code"] != "InvalidPermission.Duplicate":
        raise
    print("Port 5432 is already open")

In [ ]:
def database_url(host, password):
    """The address of the database, with its password, requiring TLS."""
    return f"postgresql://{DB_USER}:{password}@{host}:5432/postgres?sslmode=require"


load_dotenv(override=True)
saved_url = os.getenv("AWS_DATABASE_URL")
db_password = urlsplit(saved_url).password if saved_url else secrets.token_urlsafe(24)

try:
    db = rds.describe_db_instances(DBInstanceIdentifier=DB_INSTANCE)["DBInstances"][0]
    print(f"Database {DB_INSTANCE} already exists ({db['DBInstanceStatus']})")
    if not saved_url:
        # Its password isn't in .env, so give it a new one that is.
        set_key(
            ENV_FILE,
            "AWS_DATABASE_URL",
            database_url("pending", db_password),
            quote_mode="never",
        )
        rds.modify_db_instance(
            DBInstanceIdentifier=DB_INSTANCE,
            MasterUserPassword=db_password,
            ApplyImmediately=True,
        )
        print("Gave it a new password, kept in .env")
except rds.exceptions.DBInstanceNotFoundFault:
    set_key(
        ENV_FILE,
        "AWS_DATABASE_URL",
        database_url("pending", db_password),
        quote_mode="never",
    )
    version = rds.describe_db_engine_versions(
        Engine="postgres", EngineVersion="17", DefaultOnly=True
    )["DBEngineVersions"][0]["EngineVersion"]
    rds.create_db_instance(
        DBInstanceIdentifier=DB_INSTANCE,
        DBInstanceClass="db.t4g.micro",
        Engine="postgres",
        EngineVersion=version,
        AllocatedStorage=20,
        StorageType="gp3",
        MultiAZ=False,
        BackupRetentionPeriod=1,
        PubliclyAccessible=True,
        VpcSecurityGroupIds=[db_group_id],
        MasterUsername=DB_USER,
        MasterUserPassword=db_password,
        Tags=FDE_TAG_LIST,
    )
    print(
        f"Creating {DB_INSTANCE} on PostgreSQL {version}, which takes 5 to 10 minutes"
    )

rds.get_waiter("db_instance_available").wait(
    DBInstanceIdentifier=DB_INSTANCE, WaiterConfig={"Delay": 30, "MaxAttempts": 60}
)
db_host = rds.describe_db_instances(DBInstanceIdentifier=DB_INSTANCE)["DBInstances"][0][
    "Endpoint"
]["Address"]
set_key(
    ENV_FILE, "AWS_DATABASE_URL", database_url(db_host, db_password), quote_mode="never"
)
print(f"{DB_INSTANCE} is available at {db_host}, and AWS_DATABASE_URL is in .env")

### seeding the database

On Lambda the app never makes its tables, so we seed the database once from here: `backend.seed`, as
`npm run seed` runs it, with `DATABASE_URL` set to `AWS_DATABASE_URL`. It doesn't read `.env`, so your
local `DATABASE_URL` can't take its place. Seeding again starts the queue afresh. Then we count the claims
and check that the connection uses TLS.

In [ ]:
def seed_rds():
    """Seed the database on RDS from data/, as npm run seed does locally."""
    load_dotenv(override=True)
    seeded = subprocess.run(  # noqa: S603 (a fixed command)
        [shutil.which("uv"), "run", "-m", "backend.seed"],
        cwd="../..",
        env=os.environ | {"DATABASE_URL": os.environ["AWS_DATABASE_URL"]},
        capture_output=True,
        text=True,
        check=False,
    )
    print(seeded.stdout.strip() or seeded.stderr[-2000:])
    seeded.check_returncode()


def check_rds():
    """Count the claims on RDS, and show that the connection uses TLS."""
    with psycopg.connect(os.environ["AWS_DATABASE_URL"]) as connection:
        claims = connection.execute(
            "SELECT count(*) FROM undercover.claims"
        ).fetchone()[0]
        tls = connection.execute(
            "SELECT ssl FROM pg_stat_ssl WHERE pid = pg_backend_pid()"
        ).fetchone()[0]
        # rds.force_ssl may be hidden from this user; None means it could not be read.
        force_ssl = connection.execute(
            "SELECT current_setting('rds.force_ssl', true)"
        ).fetchone()[0]
    print(
        f"Claims: {claims}. This connection uses TLS: {tls}. rds.force_ssl: {force_ssl}"
    )


seed_rds()
check_rds()

### the handlers' sign-in with Amazon Cognito

A Cognito user pool, `undercover-handlers`, signs the handlers in to the deployed dashboard, in place of
Supabase Auth. Self sign-up is off, and a handler's email is their user name. Its app client,
`undercover-dashboard`, allows `USER_PASSWORD_AUTH` and has no secret, since the dashboard runs in a
browser that can't keep one. The three handlers from `data/handlers.json` get `HANDLER_PASSWORD` as their
password, permanent, so they're never asked to change it. The password rule is only a length of at least
8, so `HANDLER_PASSWORD` fits. Cognito is free at three users.

The pool's and the client's ids go into `.env` as `AWS_COGNITO_USER_POOL_ID` and `AWS_COGNITO_CLIENT_ID`,
not as the dashboard's `VITE_COGNITO_...` settings: `npm run dev` reads `.env`, and on your laptop the
dashboard keeps signing in through local Supabase.

In [ ]:
pools = [
    pool
    for page in cognito.get_paginator("list_user_pools").paginate(MaxResults=60)
    for pool in page["UserPools"]
    if pool["Name"] == USER_POOL_NAME
]
if pools:
    user_pool_id = pools[0]["Id"]
    print(f"User pool {USER_POOL_NAME} already exists")
else:
    user_pool_id = cognito.create_user_pool(
        PoolName=USER_POOL_NAME,
        UsernameAttributes=["email"],
        AdminCreateUserConfig={"AllowAdminCreateUserOnly": True},
        Policies={
            "PasswordPolicy": {
                "MinimumLength": 8,
                "RequireUppercase": False,
                "RequireLowercase": False,
                "RequireNumbers": False,
                "RequireSymbols": False,
            }
        },
        DeletionProtection="INACTIVE",
        UserPoolTags=FDE_TAGS,
    )["UserPool"]["Id"]
    print(f"Created user pool {USER_POOL_NAME}")

clients = [
    client
    for page in cognito.get_paginator("list_user_pool_clients").paginate(
        UserPoolId=user_pool_id
    )
    for client in page["UserPoolClients"]
    if client["ClientName"] == APP_CLIENT_NAME
]
if clients:
    client_id = clients[0]["ClientId"]
    print(f"App client {APP_CLIENT_NAME} already exists")
else:
    client_id = cognito.create_user_pool_client(
        UserPoolId=user_pool_id,
        ClientName=APP_CLIENT_NAME,
        GenerateSecret=False,
        ExplicitAuthFlows=["ALLOW_USER_PASSWORD_AUTH", "ALLOW_REFRESH_TOKEN_AUTH"],
        PreventUserExistenceErrors="ENABLED",
    )["UserPoolClient"]["ClientId"]
    print(f"Created app client {APP_CLIENT_NAME}")

set_key(ENV_FILE, "AWS_COGNITO_USER_POOL_ID", user_pool_id, quote_mode="never")
set_key(ENV_FILE, "AWS_COGNITO_CLIENT_ID", client_id, quote_mode="never")
print("AWS_COGNITO_USER_POOL_ID and AWS_COGNITO_CLIENT_ID are in .env")

In [ ]:
handlers = json.loads(Path("../../data/handlers.json").read_text(encoding="utf-8"))
handler_password = os.environ["HANDLER_PASSWORD"]

for handler in handlers:
    try:
        cognito.admin_create_user(
            UserPoolId=user_pool_id,
            Username=handler["email"],
            UserAttributes=[
                {"Name": "email", "Value": handler["email"]},
                {"Name": "email_verified", "Value": "true"},
                {"Name": "name", "Value": handler["name"]},
            ],
            MessageAction="SUPPRESS",
        )
        print(f"Added {handler['name']}")
    except cognito.exceptions.UsernameExistsException:
        print(f"{handler['name']} is already in the pool")
    cognito.admin_set_user_password(
        UserPoolId=user_pool_id,
        Username=handler["email"],
        Password=handler_password,
        Permanent=True,
    )

Each handler signs in, as the dashboard will, with `USER_PASSWORD_AUTH`:

In [ ]:
for handler in handlers:
    reply = cognito.initiate_auth(
        ClientId=client_id,
        AuthFlow="USER_PASSWORD_AUTH",
        AuthParameters={"USERNAME": handler["email"], "PASSWORD": handler_password},
    )
    print(f"{handler['name']} signs in: {'AuthenticationResult' in reply}")

### What Lab 2 and Day 3 will read

These three settings are in `.env` now. Lab 2 and Day 3 see them and deploy the app with the database on RDS and the sign-in on Cognito. The values aren't printed, since the database's address holds its password.

In [ ]:
load_dotenv(ENV_FILE, override=True)
for name in ["AWS_DATABASE_URL", "AWS_COGNITO_USER_POOL_ID", "AWS_COGNITO_CLIENT_ID"]:
    print(f"{name}: {'in .env' if os.getenv(name) else 'missing, run the cells above'}")

## Part 3: Teardown

Run this part once you've finished Week 3's labs, or whenever you stop for more than a day: the database costs about $0.60 a day while it exists. Lab 2 and Day 3 delete their own pieces on Lambda and AgentCore.

These cells delete the database with no final snapshot and with its automated backups, then its security group and the user pool, and take the three settings out of `.env`. Each is safe to run again. Deleting the database takes about 5 minutes. Running Parts 1 and 2 again makes everything afresh, with a new password.

In [ ]:
from botocore.exceptions import ClientError

rds = session.client("rds")
ec2 = session.client("ec2")
cognito = session.client("cognito-idp")

In [ ]:
try:
    rds.delete_db_instance(
        DBInstanceIdentifier=DB_INSTANCE,
        SkipFinalSnapshot=True,
        DeleteAutomatedBackups=True,
    )
    print(f"Deleting {DB_INSTANCE}, which takes about 5 minutes")
except rds.exceptions.DBInstanceNotFoundFault:
    print(f"{DB_INSTANCE} is already deleted")
except rds.exceptions.InvalidDBInstanceStateFault:
    print(f"{DB_INSTANCE} is already being deleted")

rds.get_waiter("db_instance_deleted").wait(
    DBInstanceIdentifier=DB_INSTANCE, WaiterConfig={"Delay": 30, "MaxAttempts": 60}
)
for backup in rds.describe_db_instance_automated_backups()[
    "DBInstanceAutomatedBackups"
]:
    if backup["DBInstanceIdentifier"] == DB_INSTANCE:
        rds.delete_db_instance_automated_backup(DbiResourceId=backup["DbiResourceId"])
        print(f"Deleted an automated backup of {DB_INSTANCE}")
print(f"{DB_INSTANCE} is deleted")

In [ ]:
groups = ec2.describe_security_groups(
    Filters=[{"Name": "group-name", "Values": [DB_SECURITY_GROUP]}]
)
for group in groups["SecurityGroups"]:
    # The database's network interface can take a minute to go after the database.
    for _ in range(20):
        try:
            ec2.delete_security_group(GroupId=group["GroupId"])
            print(f"Deleted security group {DB_SECURITY_GROUP}")
            break
        except ClientError as error:
            if error.response["Error"]["Code"] != "DependencyViolation":
                raise
            print("Waiting for the database's network interface to go")
            time.sleep(15)

In [ ]:
for page in cognito.get_paginator("list_user_pools").paginate(MaxResults=60):
    for pool in page["UserPools"]:
        if pool["Name"] == USER_POOL_NAME:
            cognito.delete_user_pool(UserPoolId=pool["Id"])
            print(f"Deleted user pool {USER_POOL_NAME}")

In [ ]:
for name in ["AWS_DATABASE_URL", "AWS_COGNITO_USER_POOL_ID", "AWS_COGNITO_CLIENT_ID"]:
    if name in os.environ or name in ENV_FILE.read_text(encoding="utf-8"):
        unset_key(ENV_FILE, name)
        os.environ.pop(name, None)
print("Took AWS_DATABASE_URL and the Cognito settings out of .env")

### Anything left?

Every resource in us-east-1 still tagged `label=FDE-lab`, through the Resource Groups Tagging API, and the database, security group and user pool by name. The tagging listing can lag by a few minutes after a deletion, so run it again if something you just deleted still shows. Then check the spend with `uv run monitor.py`.

In [ ]:
tagged = [
    resource["ResourceARN"]
    for page in session.client("resourcegroupstaggingapi")
    .get_paginator("get_resources")
    .paginate(TagFilters=[{"Key": "label", "Values": ["FDE-lab"]}])
    for resource in page["ResourceTagMappingList"]
]
print(f"{'Tagged FDE-lab:':<20}{', '.join(tagged) or 'none'}")

named = {
    "RDS databases": [
        db["DBInstanceIdentifier"]
        for db in rds.describe_db_instances()["DBInstances"]
        if db["DBInstanceIdentifier"].startswith("undercover")
    ],
    "Security groups": [
        group["GroupName"]
        for group in ec2.describe_security_groups(
            Filters=[{"Name": "group-name", "Values": ["undercover*"]}]
        )["SecurityGroups"]
    ],
    "Cognito user pools": [
        pool["Name"]
        for page in cognito.get_paginator("list_user_pools").paginate(MaxResults=60)
        for pool in page["UserPools"]
        if pool["Name"].startswith("undercover")
    ],
}
for label, names in named.items():
    print(f"{label + ':':<20}{', '.join(names) or 'none'}")